# 08. Representation Concatenation & Gated Dual Encoder

**Paper section:** §8.2 Representation concatenation; §8.3 Gated dual encoder (Table 8).
**What it computes:** Tests whether concatenating Latin and Unicode features helps POS classification, and whether a learned per-input gating network (gated dual encoder) outperforms a simple baseline transformer. This is the centerpiece of the language-specific-vs-script-level argument.
**Inputs:** Outputs of notebook 01; outputs of notebook 07.
**Outputs:** `outputs/table8_concatenation.csv`, `outputs/table8_gated_dual_encoder.json`, `outputs/figure4_gating_weights.png`.
**Expected runtime (CPU baseline):** ~20 min on CPU; ~5 min on GPU.

All randomness uses `SEED = 42`.

## Setup
This notebook uses shared utilities from `_setup.py`:
- `load_corpora()`: returns dict of dataframes per language
- `POS_HARMONIZATION`: dict mapping raw POS tags to unified tagset
- `BASE_PATH`: data directory (set this for your environment)

```python
from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH
```


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
try:
    import torch
    torch.manual_seed(42)
except ImportError:
    pass

from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH, set_seeds
set_seeds(42)


In [ ]:
# --- data-availability guard ------------------------------------------
missing = [l for l in ('akk', 'sux', 'elx') if l not in corpora]
if missing:
    print(f'WARNING: {missing} not loaded. Cells that depend on them will be skipped.')
    print(f'Set $CUNEI_DATA to a directory containing alltexts_AKK.csv, alltexts_SUX.csv, and the Elamite files.')
# Convenience: expose datasets dict for cells originally from the monolith.
datasets = {l: corpora[l] for l in ('akk', 'sux', 'elx') if l in corpora}
sign_dict = corpora.get('_sign_dict', {})


In [ ]:
corpora = load_corpora(BASE_PATH)


## Experiment 3c: Concatenation Experiment

Test whether combining Latin + Unicode features improves over either alone.
Run on all tasks: unified POS, grammatical POS, entity detection, entity typing.


In [ ]:
# ============================================================
# EXPERIMENT 3c: CONCATENATION (Latin + Unicode)
# ============================================================
exp3c_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  EXP 3c — {lang.upper()}: Concatenation")
    print(f"{'='*60}")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    results = {'lang': lang}

    tasks = []

    # Task: Unified POS
    counts = df['pos_unified'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    task_df = df[df['pos_unified'].isin(valid)].copy()
    if len(task_df) > 50000:
        task_df = task_df.sample(50000, random_state=SEED)
    if len(valid) >= 2:
        tasks.append(('unified_pos', task_df, 'pos_unified'))

    # Task: Grammatical POS (no entities)
    gram_df = df[df['ner_tag'] == 'O'].copy()
    counts = gram_df['pos_grammatical'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    gram_clf = gram_df[gram_df['pos_grammatical'].isin(valid)]
    if len(gram_clf) > 50000:
        gram_clf = gram_clf.sample(50000, random_state=SEED)
    if len(valid) >= 2:
        tasks.append(('gram_only', gram_clf, 'pos_grammatical'))

    # Task: Entity detection
    det_df = df.copy()
    det_df['is_entity'] = (det_df['ner_tag'] != 'O').astype(str)
    if len(det_df) > 50000:
        det_df = det_df.sample(50000, random_state=SEED)
    tasks.append(('entity_detect', det_df, 'is_entity'))

    # Task: Entity typing
    ent_df = df[df['ner_tag'] != 'O'].copy()
    ent_counts = ent_df['ner_tag'].value_counts()
    ent_valid = ent_counts[ent_counts >= MIN_CLASS_COUNT].index.tolist()
    ent_clf = ent_df[ent_df['ner_tag'].isin(ent_valid)]
    if len(ent_clf) > 50000:
        ent_clf = ent_clf.sample(50000, random_state=SEED)
    if len(ent_valid) >= 2:
        tasks.append(('entity_type', ent_clf, 'ner_tag'))

    for task_name, task_df, label_col in tasks:
        print(f"\n  {task_name}: {len(task_df):,} tokens")
        texts_l = task_df['form_latin'].astype(str).values
        texts_u = task_df['form_unicode'].astype(str).values
        labels = task_df[label_col].values

        vec_l = CountVectorizer(analyzer='char', ngram_range=(1,4))
        vec_u = CountVectorizer(analyzer='char', ngram_range=(1,4))
        X_l = vec_l.fit_transform(texts_l)
        X_u = vec_u.fit_transform(texts_u)

        from scipy.sparse import hstack
        X_concat = hstack([X_l, X_u])

        clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)

        # Latin only
        s_l = cross_val_score(clf, X_l, labels, cv=cv, scoring='f1_macro')
        # Unicode only
        s_u = cross_val_score(clf, X_u, labels, cv=cv, scoring='f1_macro')
        # Concatenated
        s_c = cross_val_score(clf, X_concat, labels, cv=cv, scoring='f1_macro')

        results[f'{task_name}_latin'] = s_l.mean()
        results[f'{task_name}_unicode'] = s_u.mean()
        results[f'{task_name}_concat'] = s_c.mean()

        best = max(s_l.mean(), s_u.mean(), s_c.mean())
        print(f"    Latin:  {s_l.mean():.4f} (±{s_l.std():.4f})")
        print(f"    Unicode: {s_u.mean():.4f} (±{s_u.std():.4f})")
        print(f"    Concat:  {s_c.mean():.4f} (±{s_c.std():.4f}) {'*** BEST' if s_c.mean() == best else ''}")

    exp3c_results[lang] = results


In [ ]:
# ============================================================
# STATISTICAL SIGNIFICANCE TESTS
# ============================================================
# Paired bootstrap test on all key Latin vs Unicode and
# Concat vs Best comparisons.
#
# Add this cell after Exp 3c in your notebook.
# ============================================================

from scipy.sparse import hstack
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

def paired_bootstrap_test(y_true, preds_a, preds_b, n_bootstrap=2000, seed=42):
    """
    Paired bootstrap significance test.
    Tests whether system A is significantly different from system B.

    Args:
        y_true: Gold labels
        preds_a: Predictions from system A
        preds_b: Predictions from system B
        n_bootstrap: Number of bootstrap samples
        seed: Random seed

    Returns:
        dict with p_value, score_a, score_b, delta, ci_lower, ci_upper
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)

    score_a = f1_score(y_true, preds_a, average='macro', zero_division=0)
    score_b = f1_score(y_true, preds_b, average='macro', zero_division=0)
    observed_delta = score_a - score_b

    deltas = []
    count_b_wins = 0

    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, size=n)
        boot_true = y_true[idx]
        boot_a = preds_a[idx]
        boot_b = preds_b[idx]

        sa = f1_score(boot_true, boot_a, average='macro', zero_division=0)
        sb = f1_score(boot_true, boot_b, average='macro', zero_division=0)
        delta = sa - sb
        deltas.append(delta)

        # Two-sided: count how often the sign flips
        if observed_delta >= 0 and delta <= 0:
            count_b_wins += 1
        elif observed_delta < 0 and delta >= 0:
            count_b_wins += 1

    deltas = np.array(deltas)
    p_value = (2 * count_b_wins) / n_bootstrap  # two-sided
    p_value = min(p_value, 1.0)

    return {
        'score_a': score_a,
        'score_b': score_b,
        'delta': observed_delta,
        'p_value': p_value,
        'ci_lower': np.percentile(deltas, 2.5),
        'ci_upper': np.percentile(deltas, 97.5),
        'significant_005': p_value < 0.05,
        'significant_001': p_value < 0.01,
    }


def run_significance_for_task(texts_l, texts_u, labels, task_name, lang):
    """
    Train Latin, Unicode, and Concat models, collect predictions,
    run paired bootstrap on all pairs.
    """
    vec_l = CountVectorizer(analyzer='char', ngram_range=(1, 4))
    vec_u = CountVectorizer(analyzer='char', ngram_range=(1, 4))
    X_l = vec_l.fit_transform(texts_l)
    X_u = vec_u.fit_transform(texts_u)
    X_c = hstack([X_l, X_u])

    labels = np.array(labels)

    # Collect cross-validated predictions for each system
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    preds_l = np.empty(len(labels), dtype=labels.dtype)
    preds_u = np.empty(len(labels), dtype=labels.dtype)
    preds_c = np.empty(len(labels), dtype=labels.dtype)

    for train_idx, test_idx in cv.split(X_l, labels):
        clf_l = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, C=1.0)
        clf_u = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, C=1.0)
        clf_c = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, C=1.0)

        clf_l.fit(X_l[train_idx], labels[train_idx])
        clf_u.fit(X_u[train_idx], labels[train_idx])
        clf_c.fit(X_c[train_idx], labels[train_idx])

        preds_l[test_idx] = clf_l.predict(X_l[test_idx])
        preds_u[test_idx] = clf_u.predict(X_u[test_idx])
        preds_c[test_idx] = clf_c.predict(X_c[test_idx])

    # Run bootstrap tests
    results = {}

    # Latin vs Unicode
    test_lu = paired_bootstrap_test(labels, preds_l, preds_u)
    sig_lu = '***' if test_lu['significant_001'] else ('**' if test_lu['significant_005'] else 'ns')
    results['latin_vs_unicode'] = test_lu

    # Concat vs Latin
    test_cl = paired_bootstrap_test(labels, preds_c, preds_l)
    sig_cl = '***' if test_cl['significant_001'] else ('**' if test_cl['significant_005'] else 'ns')
    results['concat_vs_latin'] = test_cl

    # Concat vs Unicode
    test_cu = paired_bootstrap_test(labels, preds_c, preds_u)
    sig_cu = '***' if test_cu['significant_001'] else ('**' if test_cu['significant_005'] else 'ns')
    results['concat_vs_unicode'] = test_cu

    print(f"\n  {task_name}:")
    print(f"    Latin={test_lu['score_a']:.4f}  Unicode={test_lu['score_b']:.4f}  "
          f"Concat={test_cl['score_a']:.4f}")
    print(f"    Latin vs Unicode:  Δ={test_lu['delta']:+.4f}  p={test_lu['p_value']:.4f}  "
          f"CI=[{test_lu['ci_lower']:+.4f}, {test_lu['ci_upper']:+.4f}]  {sig_lu}")
    print(f"    Concat vs Latin:   Δ={test_cl['delta']:+.4f}  p={test_cl['p_value']:.4f}  "
          f"CI=[{test_cl['ci_lower']:+.4f}, {test_cl['ci_upper']:+.4f}]  {sig_cl}")
    print(f"    Concat vs Unicode: Δ={test_cu['delta']:+.4f}  p={test_cu['p_value']:.4f}  "
          f"CI=[{test_cu['ci_lower']:+.4f}, {test_cu['ci_upper']:+.4f}]  {sig_cu}")

    return results


# ============================================================
# RUN ALL SIGNIFICANCE TESTS
# ============================================================
all_sig_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  SIGNIFICANCE TESTS — {lang.upper()}")
    print(f"{'='*60}")

    lang_results = {}

    # ── Unified POS ──
    counts = df['pos_unified'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    task_df = df[df['pos_unified'].isin(valid)].copy()
    if len(task_df) > 10000:
        task_df = task_df.sample(10000, random_state=SEED)

    if len(valid) >= 2:
        lang_results['unified_pos'] = run_significance_for_task(
            task_df['form_latin'].astype(str).values,
            task_df['form_unicode'].astype(str).values,
            task_df['pos_unified'].values,
            'Unified POS', lang
        )

    # ── Grammatical POS (entities excluded) ──
    gram_df = df[df['ner_tag'] == 'O'].copy()
    counts = gram_df['pos_grammatical'].value_counts()
    valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
    gram_clf = gram_df[gram_df['pos_grammatical'].isin(valid)]
    if len(gram_clf) > 10000:
        gram_clf = gram_clf.sample(10000, random_state=SEED)

    if len(valid) >= 2:
        lang_results['gram_pos'] = run_significance_for_task(
            gram_clf['form_latin'].astype(str).values,
            gram_clf['form_unicode'].astype(str).values,
            gram_clf['pos_grammatical'].values,
            'Grammatical POS (no entities)', lang
        )

    # ── Entity detection (binary) ──
    det_df = df.copy()
    det_df['is_entity'] = (det_df['ner_tag'] != 'O').astype(str)
    if len(det_df) > 10000:
        det_df = det_df.sample(10000, random_state=SEED)

    lang_results['entity_detect'] = run_significance_for_task(
        det_df['form_latin'].astype(str).values,
        det_df['form_unicode'].astype(str).values,
        det_df['is_entity'].values,
        'Entity detection (binary)', lang
    )

    # ── Entity typing ──
    ent_df = df[df['ner_tag'] != 'O'].copy()
    ent_counts = ent_df['ner_tag'].value_counts()
    ent_valid = ent_counts[ent_counts >= MIN_CLASS_COUNT].index.tolist()
    ent_clf = ent_df[ent_df['ner_tag'].isin(ent_valid)]
    if len(ent_clf) > 10000:
        ent_clf = ent_clf.sample(10000, random_state=SEED)

    if len(ent_valid) >= 2:
        lang_results['entity_type'] = run_significance_for_task(
            ent_clf['form_latin'].astype(str).values,
            ent_clf['form_unicode'].astype(str).values,
            ent_clf['ner_tag'].values,
            'Entity typing', lang
        )

    all_sig_results[lang] = lang_results

# ============================================================
# SUMMARY TABLE
# ============================================================
print(f"\n{'='*60}")
print(f"  SIGNIFICANCE SUMMARY")
print(f"{'='*60}")
print(f"\n  *** = p < 0.01,  ** = p < 0.05,  ns = not significant\n")
print(f"  {'Comparison':<25s} {'Task':<20s} {'AKK':>8s} {'SUX':>8s} {'ELX':>8s}")
print(f"  {'-'*25} {'-'*20} {'-'*8} {'-'*8} {'-'*8}")

for task in ['unified_pos', 'gram_pos', 'entity_detect', 'entity_type']:
    for comp in ['latin_vs_unicode', 'concat_vs_latin', 'concat_vs_unicode']:
        row = []
        for lang in ['akk', 'sux', 'elx']:
            if lang in all_sig_results and task in all_sig_results[lang]:
                r = all_sig_results[lang][task].get(comp, {})
                if r:
                    p = r['p_value']
                    sig = '***' if p < 0.01 else ('**' if p < 0.05 else 'ns')
                    row.append(f"{r['delta']:+.3f}{sig}")
                else:
                    row.append('—')
            else:
                row.append('—')

        comp_label = comp.replace('_', ' ').title()
        task_label = task.replace('_', ' ').title()
        print(f"  {comp_label:<25s} {task_label:<20s} {row[0]:>8s} {row[1]:>8s} {row[2]:>8s}")
    print()


## Experiment 6: Gated Dual Encoder

In [ ]:
# ============================================================
# GATED DUAL-ENCODER: Learns per-input representation weighting
# ============================================================
# Adds methodological contribution: a simple model that learns
# WHEN to trust Latin vs Unicode for each input.
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

MIN_CLASS_COUNT = 20


class GatedDualEncoder(nn.Module):
    """
    Dual-encoder with learned gating between Latin and Unicode representations.

    Each input is encoded by two separate character-level BiLSTMs.
    A learned gate determines per-input how much to trust each representation:

        h_final = g * h_latin + (1 - g) * h_unicode
        g = sigmoid(W_gate · [h_latin; h_unicode] + b_gate)

    The gate value g is interpretable: g → 1 means Latin-dominant,
    g → 0 means Unicode-dominant.
    """
    def __init__(self, vocab_size_l, vocab_size_u, embed_dim, hidden_dim, n_classes, dropout=0.3):
        super().__init__()
        # Latin encoder
        self.embed_l = nn.Embedding(vocab_size_l, embed_dim, padding_idx=0)
        self.lstm_l = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                              bidirectional=True, batch_first=True, dropout=dropout)

        # Unicode encoder
        self.embed_u = nn.Embedding(vocab_size_u, embed_dim, padding_idx=0)
        self.lstm_u = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                              bidirectional=True, batch_first=True, dropout=dropout)

        # Gate: takes concatenated hidden states, outputs scalar per sample
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

        # Classifier
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, n_classes)

    def encode(self, x, embed, lstm):
        emb = self.dropout(embed(x))
        _, (h, _) = lstm(emb)
        return torch.cat([h[-2], h[-1]], dim=1)  # concat forward/backward

    def forward(self, x_latin, x_unicode):
        h_l = self.encode(x_latin, self.embed_l, self.lstm_l)
        h_u = self.encode(x_unicode, self.embed_u, self.lstm_u)

        # Compute gate
        g = self.gate(torch.cat([h_l, h_u], dim=1))  # (batch, 1)

        # Gated combination
        h_combined = g * h_l + (1 - g) * h_u

        return self.fc(self.dropout(h_combined)), g


def build_char_vocab(texts):
    char2idx = {'<pad>': 0, '<unk>': 1}
    for t in texts:
        for ch in str(t):
            if ch not in char2idx:
                char2idx[ch] = len(char2idx)
    return char2idx


def encode_texts(texts, char2idx, max_len):
    X = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, t in enumerate(texts):
        for j, ch in enumerate(str(t)[:max_len]):
            X[i, j] = char2idx.get(ch, 1)
    return X


# ============================================================
# RUN GATED MODEL ACROSS ALL LANGUAGES AND TASKS
# ============================================================

EMBED_DIM = 64
HIDDEN_DIM = 64
DROPOUT = 0.3
LR = 0.001
BATCH_SIZE = 64
N_EPOCHS = 25
MAX_LEN = 80

gated_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  GATED DUAL-ENCODER — {lang.upper()}")
    print(f"{'='*60}")

    for pos_col, task_name in [('pos_unified', 'unified_pos'),
                                ('pos_grammatical', 'gram_pos')]:
        counts = df[pos_col].value_counts()
        valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
        task_df = df[df[pos_col].isin(valid)].copy()

        if len(task_df) > 10000:
            task_df = task_df.sample(10000, random_state=SEED)

        if len(valid) < 2:
            continue

        texts_l = task_df['form_latin'].astype(str).tolist()
        texts_u = task_df['form_unicode'].astype(str).tolist()

        le = LabelEncoder()
        labels = le.fit_transform(task_df[pos_col])
        n_classes = len(le.classes_)

        # Build vocabs
        vocab_l = build_char_vocab(texts_l)
        vocab_u = build_char_vocab(texts_u)

        X_l = encode_texts(texts_l, vocab_l, MAX_LEN)
        X_u = encode_texts(texts_u, vocab_u, MAX_LEN)

        # Cross-validation
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        fold_scores = []
        all_gates = []
        all_labels_for_gates = []
        all_sign_counts = []

        for fold, (train_idx, val_idx) in enumerate(cv.split(X_l, labels)):
            Xl_tr = torch.LongTensor(X_l[train_idx]).to(device)
            Xu_tr = torch.LongTensor(X_u[train_idx]).to(device)
            y_tr = torch.LongTensor(labels[train_idx]).to(device)

            Xl_va = torch.LongTensor(X_l[val_idx]).to(device)
            Xu_va = torch.LongTensor(X_u[val_idx]).to(device)
            y_va = labels[val_idx]

            model = GatedDualEncoder(
                len(vocab_l), len(vocab_u), EMBED_DIM, HIDDEN_DIM, n_classes, DROPOUT
            ).to(device)

            optimizer = torch.optim.Adam(model.parameters(), lr=LR)
            # Class weights
            wts = torch.FloatTensor(
                [1.0 / max((labels[train_idx] == c).sum(), 1) for c in range(n_classes)]
            ).to(device)
            criterion = nn.CrossEntropyLoss(weight=wts)

            # Train
            train_ds = TensorDataset(Xl_tr, Xu_tr, y_tr)
            loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

            model.train()
            for epoch in range(N_EPOCHS):
                for xl_b, xu_b, yb in loader:
                    optimizer.zero_grad()
                    logits, g = model(xl_b, xu_b)
                    loss = criterion(logits, yb)
                    loss.backward()
                    optimizer.step()

            # Evaluate
            model.eval()
            with torch.no_grad():
                all_preds, all_g = [], []
                for k in range(0, len(Xl_va), 256):
                    logits, g = model(Xl_va[k:k+256], Xu_va[k:k+256])
                    all_preds.append(logits.argmax(dim=1).cpu().numpy())
                    all_g.append(g.cpu().numpy())

                preds = np.concatenate(all_preds)
                gates = np.concatenate(all_g).flatten()

            f1 = f1_score(y_va, preds, average='macro')
            fold_scores.append(f1)

            # Collect gate values for analysis
            all_gates.extend(gates)
            all_labels_for_gates.extend(le.inverse_transform(y_va))
            all_sign_counts.extend([len(texts_u[i].split()) for i in val_idx])

            del model, Xl_tr, Xu_tr, Xl_va, Xu_va
            torch.cuda.empty_cache()

        mean_f1 = np.mean(fold_scores)
        print(f"\n  {task_name}: F1 = {mean_f1:.4f} (±{np.std(fold_scores):.4f})")

        gated_results.setdefault(lang, {})[task_name] = mean_f1

        # ── Gate Analysis: by POS category ──
        print(f"\n  Gate values by POS (g→1 = Latin, g→0 = Unicode):")
        gates_arr = np.array(all_gates)
        labels_arr = np.array(all_labels_for_gates)
        signs_arr = np.array(all_sign_counts)

        for pos in sorted(set(labels_arr)):
            mask = labels_arr == pos
            if mask.sum() >= 10:
                mean_g = gates_arr[mask].mean()
                direction = "← Latin" if mean_g > 0.55 else ("→ Unicode" if mean_g < 0.45 else "balanced")
                print(f"    {pos:<10s}: g={mean_g:.3f} {direction}  (n={mask.sum()})")

        # ── Gate Analysis: by sign count ──
        print(f"\n  Gate values by sign count:")
        for lo, hi, label in [(1, 1, '1 sign'), (2, 2, '2 signs'), (3, 3, '3 signs'),
                               (4, 5, '4-5'), (6, 100, '6+')]:
            mask = (signs_arr >= lo) & (signs_arr <= hi)
            if mask.sum() >= 10:
                mean_g = gates_arr[mask].mean()
                print(f"    {label:>7s}: g={mean_g:.3f}  (n={mask.sum()})")


# ── Comparison table ──
print(f"\n{'='*60}")
print(f"  MODEL COMPARISON")
print(f"{'='*60}")
print(f"\n  {'Task':<15s} {'Lang':<6s} {'Latin':>8s} {'Unicode':>8s} {'Concat':>8s} {'Gated':>8s}")
print(f"  {'-'*15} {'-'*6} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")

# You'll need to have exp3c_results available, or manually enter the values
for lang in ['akk', 'sux', 'elx']:
    if lang not in gated_results:
        continue
    for task in ['unified_pos', 'gram_pos']:
        gated_f1 = gated_results[lang].get(task, 0)
        print(f"  {task:<15s} {lang.upper():<6s} {'---':>8s} {'---':>8s} {'---':>8s} {gated_f1:>8.4f}")

## Experiment 6: Baseline Transformer

In [ ]:
# ============================================================
# CHARACTER TRANSFORMER BASELINE
# ============================================================
# Tests whether the complementarity effect persists under
# a modern neural architecture (not just linear LR).
#
# Simple 2-layer character Transformer encoder trained from
# scratch. Runs Latin vs Unicode vs Concat on unified POS
# and grammatical POS for all three languages.
#
# Prerequisites: cells 1-13 (data loading), MIN_CLASS_COUNT=20
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import math

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

MIN_CLASS_COUNT = 20


class CharTransformerClassifier(nn.Module):
    """
    Simple character-level Transformer encoder for text classification.
    2-layer, 4-head, trained from scratch.
    """
    def __init__(self, vocab_size, n_classes, embed_dim=128, n_heads=4,
                 n_layers=2, max_len=100, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=256,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, n_classes)
        self.embed_dim = embed_dim

    def forward(self, x):
        # x: (batch, seq_len)
        mask = (x == 0)  # padding mask

        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        emb = self.embed(x) * math.sqrt(self.embed_dim) + self.pos_embed(positions)
        emb = self.dropout(emb)

        out = self.transformer(emb, src_key_padding_mask=mask)

        # Mean pooling over non-padding tokens
        mask_expanded = (~mask).unsqueeze(-1).float()
        pooled = (out * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)

        return self.fc(self.dropout(pooled))


def build_vocab(texts):
    char2idx = {'<pad>': 0, '<unk>': 1, '<sep>': 2}
    for t in texts:
        for ch in str(t):
            if ch not in char2idx:
                char2idx[ch] = len(char2idx)
    return char2idx


def encode(texts, vocab, max_len):
    X = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, t in enumerate(texts):
        for j, ch in enumerate(str(t)[:max_len]):
            X[i, j] = vocab.get(ch, 1)
    return X


def run_transformer_cv(X, labels_enc, n_classes, vocab_size, lang, repr_name,
                        max_len=100, n_splits=5, n_epochs=20, batch_size=64, lr=0.001):
    """Train and evaluate character Transformer with cross-validation."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, labels_enc)):
        X_tr = torch.LongTensor(X[train_idx]).to(device)
        y_tr = torch.LongTensor(labels_enc[train_idx]).to(device)
        X_va = torch.LongTensor(X[val_idx]).to(device)
        y_va = labels_enc[val_idx]

        model = CharTransformerClassifier(
            vocab_size=vocab_size, n_classes=n_classes,
            embed_dim=128, n_heads=4, n_layers=2, max_len=max_len, dropout=0.3
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

        wts = torch.FloatTensor(
            [1.0 / max((labels_enc[train_idx] == c).sum(), 1) for c in range(n_classes)]
        ).to(device)
        criterion = nn.CrossEntropyLoss(weight=wts)

        loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

        model.train()
        for epoch in range(n_epochs):
            for xb, yb in loader:
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

        model.eval()
        with torch.no_grad():
            all_preds = []
            for k in range(0, len(X_va), 256):
                preds = model(X_va[k:k+256]).argmax(dim=1).cpu().numpy()
                all_preds.append(preds)
            preds = np.concatenate(all_preds)

        f1 = f1_score(y_va, preds, average='macro')
        fold_scores.append(f1)

        del model, X_tr, y_tr, X_va
        torch.cuda.empty_cache()

    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores)
    print(f"    {repr_name}: {mean_f1:.4f} (±{std_f1:.4f})")
    return mean_f1


# ============================================================
# RUN TRANSFORMER ON ALL LANGUAGES
# ============================================================

MAX_LEN = 80
N_EPOCHS = 20
transformer_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  CHAR TRANSFORMER — {lang.upper()}")
    print(f"{'='*60}")

    for pos_col, task_name in [('pos_unified', 'unified_pos'),
                                ('pos_grammatical', 'gram_pos')]:
        counts = df[pos_col].value_counts()
        valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
        task_df = df[df[pos_col].isin(valid)].copy()

        if len(task_df) > 10000:
            task_df = task_df.sample(10000, random_state=SEED)

        if len(valid) < 2:
            continue

        print(f"\n  {task_name} ({len(task_df):,} tokens, {len(valid)} classes)")

        le = LabelEncoder()
        labels_enc = le.fit_transform(task_df[pos_col])
        n_classes = len(le.classes_)

        texts_l = task_df['form_latin'].astype(str).tolist()
        texts_u = task_df['form_unicode'].astype(str).tolist()
        # Concat: "latin_text [SEP] unicode_text"
        texts_c = [f"{l} \x00 {u}" for l, u in zip(texts_l, texts_u)]

        # Build combined vocab (covers all three conditions)
        all_texts = texts_l + texts_u + texts_c
        vocab = build_vocab(all_texts)

        X_l = encode(texts_l, vocab, MAX_LEN)
        X_u = encode(texts_u, vocab, MAX_LEN)
        X_c = encode(texts_c, vocab, min(MAX_LEN * 2, 160))

        # Latin
        f1_l = run_transformer_cv(X_l, labels_enc, n_classes, len(vocab),
                                   lang, 'Latin', MAX_LEN, n_epochs=N_EPOCHS)

        # Unicode
        f1_u = run_transformer_cv(X_u, labels_enc, n_classes, len(vocab),
                                   lang, 'Unicode', MAX_LEN, n_epochs=N_EPOCHS)

        # Concat
        f1_c = run_transformer_cv(X_c, labels_enc, n_classes, len(vocab),
                                   lang, 'Concat', min(MAX_LEN * 2, 160), n_epochs=N_EPOCHS)

        best_single = max(f1_l, f1_u)
        gain = f1_c - best_single
        sig = '***' if gain > 0.01 else ('+' if gain > 0 else '-')

        transformer_results.setdefault(lang, {})[task_name] = {
            'latin': f1_l, 'unicode': f1_u, 'concat': f1_c,
            'gain': gain, 'sig': sig
        }

        print(f"    Concat gain: {gain:+.4f} {sig}")

# ============================================================
# SUMMARY COMPARISON: LR vs Transformer
# ============================================================
print(f"\n{'='*60}")
print(f"  TRANSFORMER vs LR COMPARISON")
print(f"{'='*60}")
print(f"\n  Does concatenation complementarity persist under a Transformer?")
print(f"\n  {'Task':<15s} {'Lang':<5s} {'LR-L':>7s} {'LR-U':>7s} {'LR-C':>7s} {'TF-L':>7s} {'TF-U':>7s} {'TF-C':>7s} {'LR Δ':>7s} {'TF Δ':>7s}")
print(f"  {'-'*15} {'-'*5} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7}")

# LR results (enter your actual values here or pull from exp3c_results)
lr_results = {
    'akk': {
        'unified_pos': {'latin': 0.693, 'unicode': 0.621, 'concat': 0.712},
        'gram_pos': {'latin': 0.669, 'unicode': 0.657, 'concat': 0.682},
    },
    'sux': {
        'unified_pos': {'latin': 0.847, 'unicode': 0.750, 'concat': 0.870},
        'gram_pos': {'latin': 0.872, 'unicode': 0.832, 'concat': 0.888},
    },
    'elx': {
        'unified_pos': {'latin': 0.636, 'unicode': 0.659, 'concat': 0.666},
        'gram_pos': {'latin': 0.615, 'unicode': 0.634, 'concat': 0.642},
    },
}

concat_wins = 0
total_comparisons = 0

for lang in ['akk', 'sux', 'elx']:
    if lang not in transformer_results:
        continue
    for task in ['unified_pos', 'gram_pos']:
        if task not in transformer_results[lang]:
            continue

        tf = transformer_results[lang][task]
        lr = lr_results.get(lang, {}).get(task, {})

        if not lr:
            continue

        lr_gain = lr['concat'] - max(lr['latin'], lr['unicode'])
        tf_gain = tf['concat'] - max(tf['latin'], tf['unicode'])

        total_comparisons += 1
        if tf_gain > 0:
            concat_wins += 1

        print(f"  {task:<15s} {lang.upper():<5s} "
              f"{lr['latin']:>7.3f} {lr['unicode']:>7.3f} {lr['concat']:>7.3f} "
              f"{tf['latin']:>7.3f} {tf['unicode']:>7.3f} {tf['concat']:>7.3f} "
              f"{lr_gain:>+7.3f} {tf_gain:>+7.3f}")

print(f"\n  Concat wins under Transformer: {concat_wins}/{total_comparisons}")
if concat_wins == total_comparisons:
    print(f"  → Complementarity effect persists across architectures!")
elif concat_wins > total_comparisons // 2:
    print(f"  → Complementarity effect mostly persists.")
else:
    print(f"  → Complementarity effect may be architecture-dependent.")